In [ ]:
# ! pip install datasets==2.21.0
# ! pip install evaluate -U
# ! pip install transformers -U

from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from evaluate import load
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
#  You can install and import any other libraries if needed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 2.12.0
    Uninstalling datasets-2.12.0:
      Successfully uninstalled datasets-2.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [2]:
# Some Chinese punctuations will be tokenized as [UNK], so we replace them with English ones
token_replacement = [
    ["：" , ":"],
    ["，" , ","],
    ["“" , "\""],
    ["”" , "\""],
    ["？" , "?"],
    ["……" , "..."],
    ["！" , "!"]
]

In [ ]:
tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
class SemevalDataset(Dataset):
    def __init__(self, split="train") -> None:
        super().__init__()
        assert split in ["train", "validation", "test"]
        self.data = load_dataset(
            "sem_eval_2014_task_1",
            split=split,
            cache_dir="./cache/"  # <-- I removed trust_remote_code=True
        ).to_list()

    def __getitem__(self, index):
        d = self.data[index]
        # Replace Chinese punctuations with English ones
        for k in ["premise", "hypothesis"]:
            # Check if the key exists and is not None
            if d[k]:
                for tok in token_replacement:
                    d[k] = d[k].replace(tok[0], tok[1])
        return d

    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Using the latest cached version of the dataset since sem_eval_2014_task_1 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at cache/sem_eval_2014_task_1/default/1.0.0/05e094e84ece42e036799e05c4d909ee68f6df8f82f60f9302b8ef15bb9de478 (last modified on Sat Nov  8 13:14:41 2025).


Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [14]:
# Define the hyperparameters
# You can modify these values if needed
lr = 3e-5
epochs = 8 # original: 3
train_batch_size = 8
validation_batch_size = 8

In [8]:
# TODO1: Create batched data for DataLoader
# `collate_fn` is a function that defines how the data batch should be packed.
# This function will be called in the DataLoader to pack the data batch.

# With AI support
def collate_fn(batch):
    # TODO1-1: Implement the collate_fn function
    # Write your code here
    # The input parameter is a data batch (tuple), and this function packs it into tensors.
    # Use tokenizer to pack tokenize and pack the data and its corresponding labels.
    # Return the data batch and labels for each sub-task.

    # 1. Separate premises, hypotheses, and labels
    premises = [item['premise'] for item in batch]
    hypotheses = [item['hypothesis'] for item in batch]

    # These are your two separate sets of labels for the multi-output task
    labels_entailment = [item['entailment_judgment'] for item in batch]
    labels_relatedness = [item['relatedness_score'] for item in batch]

    # 2. Tokenize the text pairs
    # The tokenizer will process the pairs, pad them to the longest
    # sequence in the batch, and return PyTorch tensors.
    inputs = tokenizer(
        premises,
        hypotheses,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # 3. Convert labels to tensors
    # Task 1: Entailment (Classification) - use torch.long
    labels_entailment = torch.tensor(labels_entailment, dtype=torch.long)
    # Task 2: Relatedness (Regression) - use torch.float
    labels_relatedness = torch.tensor(labels_relatedness, dtype=torch.float)

    # 4. Return a dictionary
    # This format is easy to use in the training loop
    return {
        "inputs": inputs,
        "labels_entailment": labels_entailment,
        "labels_relatedness": labels_relatedness
    }

# --- Instantiate Datasets ---
train_dataset = SemevalDataset(split="train")
validation_dataset = SemevalDataset(split="validation")
test_dataset = SemevalDataset(split="test")


# TODO1-2: Define your DataLoader
dl_train = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    collate_fn=collate_fn
)
dl_validation = DataLoader(
    validation_dataset,
    batch_size=validation_batch_size,
    shuffle=False,  # No need to shuffle validation data
    collate_fn=collate_fn
)
dl_test = DataLoader(
    test_dataset,
    batch_size=validation_batch_size, # Can reuse validation_batch_size
    shuffle=False,  # No need to shuffle test data
    collate_fn=collate_fn
)

# Test
# This code will grab one batch from your new dl_train and print its shape
try:
    batch_sample = next(iter(dl_train))
    print("\n--- DataLoader Test Success! ---")
    print("Input shapes:")
    print("  input_ids:", batch_sample['inputs']['input_ids'].shape)
    print("  attention_mask:", batch_sample['inputs']['attention_mask'].shape)
    print("\nLabel shapes:")
    print("  entailment labels:", batch_sample['labels_entailment'].shape)
    print("  relatedness labels:", batch_sample['labels_relatedness'].shape)
except Exception as e:
    print(f"\n--- DataLoader Test Failed ---")
    print(f"Error: {e}")

Using the latest cached version of the dataset since sem_eval_2014_task_1 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at cache/sem_eval_2014_task_1/default/1.0.0/05e094e84ece42e036799e05c4d909ee68f6df8f82f60f9302b8ef15bb9de478 (last modified on Sat Nov  8 13:14:41 2025).
Using the latest cached version of the dataset since sem_eval_2014_task_1 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at cache/sem_eval_2014_task_1/default/1.0.0/05e094e84ece42e036799e05c4d909ee68f6df8f82f60f9302b8ef15bb9de478 (last modified on Sat Nov  8 13:14:41 2025).
Using the latest cached version of the dataset since sem_eval_2014_task_1 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at cache/sem_eval_2014_task_1/default/1.0.0/05e094e84ece42e036799e05c4d909ee68f6df8f82f60f9302b8ef15bb9de478 (last modified on Sat Nov  8 13:14:41 2025).



--- DataLoader Test Success! ---
Input shapes:
  input_ids: torch.Size([8, 38])
  attention_mask: torch.Size([8, 38])

Label shapes:
  entailment labels: torch.Size([8])
  relatedness labels: torch.Size([8])


In [9]:
# TODO2: Construct your model
# With AI support
class MultiLabelModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Write your code here
        # Define what modules you will use in the model
        # Please use "google-bert/bert-base-uncased" model (https://huggingface.co/google-bert/bert-base-uncased)
        # Besides the base model, you may design additional architectures by incorporating linear layers, activation functions, or other neural components.
        # Remark: The use of any additional pretrained language models is not permitted.

        # 1. Load the BERT base model
        # We use "google-bert/bert-base-uncased" as requested
        self.bert = BertModel.from_pretrained(
            "google-bert/bert-base-uncased",
            cache_dir="./cache/"
        )

        # Get the hidden size of the BERT model (which is 768)
        hidden_size = self.bert.config.hidden_size

        # 2. Add a dropout layer for regularization
        # We use the same dropout probability as the BERT model's config
        self.dropout = torch.nn.Dropout(self.bert.config.hidden_dropout_prob)

        # 3. Define the two "heads" for our multi-output task

        # Head 1: Classification for Entailment
        # The dataset has 3 classes: (0: neither, 1: entailment, 2: contradiction)
        # So the output dimension is 3.
        self.classification_head = torch.nn.Linear(hidden_size, 3)

        # Head 2: Regression for Relatedness Score
        # The output is a single continuous value
        # So the output dimension is 1.
        self.regression_head = torch.nn.Linear(hidden_size, 1)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        # Write your code here
        # Forward pass

        # 1. Pass the inputs through the base BERT model
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # 2. Get the `pooler_output`
        # This is the representation of the [CLS] token (shape: [batch_size, 768])
        # It's suitable for sentence-level classification/regression.
        pooled_output = outputs[1]

        # 3. Apply dropout
        pooled_output = self.dropout(pooled_output)

        # 4. Pass the pooled output through each head

        # Output for the classification task
        # Shape: [batch_size, 3]
        logits_classification = self.classification_head(pooled_output)

        # Output for the regression task
        # Shape: [batch_size, 1]
        output_regression = self.regression_head(pooled_output)

        # Squeeze the regression output to be shape: [batch_size]
        # This makes it easier to compare with the labels
        output_regression = output_regression.squeeze(-1)

        # 5. Return both outputs in a dictionary
        return {
            "logits_classification": logits_classification,
            "output_regression": output_regression
        }

# --- Instantiate and test the model ---
model = MultiLabelModel().to(device)

# --- Test with one batch from the DataLoader ---
# This will confirm the model can process the data
try:
    batch_sample = next(iter(dl_train))

    # Move batch to the same device as the model
    batch_inputs = batch_sample['inputs'].to(device)

    # Perform a forward pass
    with torch.no_grad(): # No need to calculate gradients for this test
        outputs = model(**batch_inputs)

    print("\n--- Model Test Success! ---")
    print("Model output shapes:")
    print("  Classification logits:", outputs['logits_classification'].shape)
    print("  Regression output:", outputs['output_regression'].shape)

    print("\nTarget label shapes (for comparison):")
    print("  Entailment labels:", batch_sample['labels_entailment'].shape)
    print("  Relatedness labels:", batch_sample['labels_relatedness'].shape)

except Exception as e:
    print(f"\n--- Model Test Failed ---")
    print(f"Error: {e}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


--- Model Test Success! ---
Model output shapes:
  Classification logits: torch.Size([8, 3])
  Regression output: torch.Size([8])

Target label shapes (for comparison):
  Entailment labels: torch.Size([8])
  Relatedness labels: torch.Size([8])


In [15]:
from transformers import get_linear_schedule_with_warmup # New Add

# TODO3: Define your optimizer and loss function
# With AI support
model = MultiLabelModel().to(device)

# TODO3-1: Define your Optimizer
optimizer = AdamW(model.parameters(), lr=lr)

# TODO3-2: Define your loss functions (you should have two)
# Write your code here
# For the multi-class classification task (entailment)
loss_fn_classification = torch.nn.CrossEntropyLoss()
# For the regression task (relatedness score)
loss_fn_regression = torch.nn.MSELoss()

# New Add
# --- Calculate Scheduler Steps ---
# Total number of training steps
num_training_steps = len(dl_train) * epochs
# 10% of steps for warmup (a common practice)
num_warmup_steps = int(0.1 * num_training_steps)

# --- Create the Scheduler ---
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)
##########

# scoring functions
psr = load("pearsonr")
acc = load("accuracy")

# --- Print confirmation ---
print("--- Optimizer, Losses, and Scheduler Defined ---")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

--- Optimizer, Losses, and Scheduler Defined ---
Optimizer: AdamW
Total training steps: 4504
Warmup steps: 450


In [11]:
# New Add
import os

# --- Create a directory to save the model ---
os.makedirs("./saved_models", exist_ok=True)

In [16]:
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    # TODO4: Write the training loop
    # With AI support
    total_loss = 0.0
    for batch in pbar:
        # Move batch data to the device (GPU)
        inputs = batch['inputs'].to(device)
        labels_entailment = batch['labels_entailment'].to(device)
        labels_relatedness = batch['labels_relatedness'].to(device)

        # 1. clear gradient
        optimizer.zero_grad()

        # 2. forward pass
        outputs = model(**inputs)

        # 3. compute loss
        loss_class = loss_fn_classification(outputs['logits_classification'], labels_entailment)
        loss_reg = loss_fn_regression(outputs['output_regression'], labels_relatedness)

        # Combine the losses. Simple addition is a good start.
        total_loss_batch = loss_class + loss_reg

        # 4. back-propagation
        total_loss_batch.backward()

        # 5. model optimization
        optimizer.step()

        # --- Update the learning rate ---
        scheduler.step()

        # --- Update progress bar with loss and LR ---
        total_loss += total_loss_batch.item()
        current_lr = scheduler.get_last_lr()[0] # Get current LR
        pbar.set_postfix({'loss': total_loss / (pbar.n + 1), 'lr': f"{current_lr:.2e}"})

    # --- Validation Loop ---
    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval()
    # TODO5: Write the evaluation loop
    # With AI support
    # Lists to store all predictions and labels
    all_class_preds = []
    all_class_labels = []
    all_reg_preds = []
    all_reg_labels = []

    with torch.no_grad(): # Disable gradient calculation for evaluation
        for batch in pbar:
            # Move batch data to the device (GPU)
            inputs = batch['inputs'].to(device)
            labels_entailment = batch['labels_entailment'].to(device)
            labels_relatedness = batch['labels_relatedness'].to(device)

            # Forward pass
            outputs = model(**inputs)

            # --- Get Predictions ---
            # For classification, get the class with the highest logit (using argmax)
            class_preds = torch.argmax(outputs['logits_classification'], dim=1)
            # For regression, the output is the prediction
            reg_preds = outputs['output_regression']

            # --- Collect predictions and labels ---
            # Move data to CPU and convert to numpy/lists for the `evaluate` library
            all_class_preds.extend(class_preds.cpu().numpy())
            all_class_labels.extend(labels_entailment.cpu().numpy())

            all_reg_preds.extend(reg_preds.cpu().numpy())
            all_reg_labels.extend(labels_relatedness.cpu().numpy())

    # --- Calculate scores ---
    # Output all the evaluation scores (PearsonCorr, Accuracy)
    pearson_corr = psr.compute(
        predictions=all_reg_preds,
        references=all_reg_labels
    )['pearsonr']

    accuracy = acc.compute(
        predictions=all_class_preds,
        references=all_class_labels
    )['accuracy']

    print(f"\n--- Epoch {ep+1} Validation Results ---")
    print(f"Accuracy (Entailment): {accuracy:.4f}")
    print(f"Pearson Corr (Relatedness): {pearson_corr:.4f}")

    # Check if this is the best model so far
    current_score = pearson_corr + accuracy
    if current_score > best_score:
        best_score = current_score
        print(f"New best score: {best_score:.4f}. Saving model...")
        torch.save(model.state_dict(), f'./saved_models/best_model.ckpt')

    print("--------------------------------------\n")

Validation epoch [1/8]: 100%|██████████| 63/63 [00:01<00:00, 52.20it/s]



--- Epoch 1 Validation Results ---
Accuracy (Entailment): 0.8180
Pearson Corr (Relatedness): 0.8322
New best score: 1.6502. Saving model...
--------------------------------------



Validation epoch [2/8]: 100%|██████████| 63/63 [00:01<00:00, 52.09it/s]



--- Epoch 2 Validation Results ---
Accuracy (Entailment): 0.8640
Pearson Corr (Relatedness): 0.8625
New best score: 1.7265. Saving model...
--------------------------------------



Validation epoch [3/8]: 100%|██████████| 63/63 [00:01<00:00, 52.49it/s]


...


--- Epoch 6 Validation Results ---
Accuracy (Entailment): 0.8560
Pearson Corr (Relatedness): 0.8739
--------------------------------------



Validation epoch [7/8]: 100%|██████████| 63/63 [00:01<00:00, 52.51it/s]



--- Epoch 7 Validation Results ---
Accuracy (Entailment): 0.8640
Pearson Corr (Relatedness): 0.8724
New best score: 1.7364. Saving model...
--------------------------------------



Validation epoch [8/8]: 100%|██████████| 63/63 [00:01<00:00, 52.48it/s]


--- Epoch 8 Validation Results ---
Accuracy (Entailment): 0.8600
Pearson Corr (Relatedness): 0.8730
--------------------------------------



In [17]:
# Load the model
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"./saved_models/best_model.ckpt", weights_only=True, map_location=device))

# Test Loop
pbar = tqdm(dl_test, desc="Test")
model.eval()

# TODO6: Write the test loop
# Write your code here
# We have loaded the best model with the highest evaluation score for you
# Please implement the test loop to evaluate the model on the test dataset
# We will have 10% of the total score for the test accuracy and pearson correlation

# With AI support
# Lists to store all test predictions and labels
all_class_preds = []
all_class_labels = []
all_reg_preds = []
all_reg_labels = []

with torch.no_grad():
    for batch in pbar:
        # Move batch data to the device (GPU)
        inputs = batch['inputs'].to(device)
        labels_entailment = batch['labels_entailment'].to(device)
        labels_relatedness = batch['labels_relatedness'].to(device)

        # Forward pass
        outputs = model(**inputs)

        # --- Get Predictions ---
        # For classification, get the class with the highest logit
        class_preds = torch.argmax(outputs['logits_classification'], dim=1)
        # For regression, the output is the prediction
        reg_preds = outputs['output_regression']

        # --- Collect predictions and labels ---
        # Move data to CPU and convert to numpy/lists for the `evaluate` library
        all_class_preds.extend(class_preds.cpu().numpy())
        all_class_labels.extend(labels_entailment.cpu().numpy())

        all_reg_preds.extend(reg_preds.cpu().numpy())
        all_reg_labels.extend(labels_relatedness.cpu().numpy())

# --- Calculate final scores ---
test_pearson_corr = psr.compute(
    predictions=all_reg_preds,
    references=all_reg_labels
)['pearsonr']

test_accuracy = acc.compute(
    predictions=all_class_preds,
    references=all_class_labels
)['accuracy']

# --- Print the final results ---
print("\n--- Final Test Results ---")
print(f"Test Accuracy (Entailment): {test_accuracy:.4f}")
print(f"Test Pearson Corr (Relatedness): {test_pearson_corr:.4f}")
print("----------------------------\n")

Test: 100%|██████████| 616/616 [00:11<00:00, 55.10it/s]


--- Final Test Results ---
Test Accuracy (Entailment): 0.8709
Test Pearson Corr (Relatedness): 0.8840
----------------------------

